In [43]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


In [32]:
df = pd.read_csv("../data/processed_data.csv", keep_default_na=False)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Spam/Ham       33716 non-null  int64 
 1   combined_text  33716 non-null  object
dtypes: int64(1), object(1)
memory usage: 526.9+ KB


In [34]:
df.head()

,Spam/Ham,combined_text
0,0,christmas tree farm pictures
1,0,vastar resources inc gary production from the ...
2,0,calpine daily gas nomination calpine daily gas...
3,0,re issue fyi see note below already done stell...
4,0,meter 7268 nov allocation fyi forwarded by lau...


In [35]:
X = df.drop(columns=["Spam/Ham"])
y = df["Spam/Ham"]

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    X["combined_text"],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words="english",
    min_df=2
)

X_train = tfidf.fit_transform(X_train)

X_test = tfidf.transform(X_test)

In [40]:
print(X_train.shape, X_test.shape)

(26972, 10000) (6744, 10000)


In [42]:
def evaluate_models(models, X_train, X_test, y_train, y_test):
    """
    Train multiple models and compare their performance.

    Parameters
    ----------
    models : dict
        Dictionary containing model name and model object.
    X_train : Training features
    X_test : Test features
    y_train : Training labels
    y_test : Test labels

    Returns
    -------
    DataFrame containing evaluation metrics.
    """

    results = []

    for name, model in models.items():

        print("=" * 70)
        print(f"Training : {name}")

        # Train
        model.fit(X_train, y_train)

        # Prediction
        y_pred = model.predict(X_test)

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append({
            "Model": name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })

        print(f"\nAccuracy : {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")
        print(f"F1 Score : {f1:.4f}")

        print("\nClassification Report")
        print(classification_report(y_test, y_pred))

        print("\nConfusion Matrix")
        print(confusion_matrix(y_test, y_pred))

    results_df = pd.DataFrame(results)

    return results_df.sort_values(
        by="F1 Score",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Naive Bayes":
        MultinomialNB(),

    "Linear SVM":
        LinearSVC(),

    "CatBoost":
        CatBoostClassifier(
            verbose=0,
            random_state=42
        )
}

In [46]:
results = evaluate_models(
    models,
    X_train,
    X_test,
    y_train,
    y_test
)

results.to_parquet(
    "../report/model_comparison_results.parquet",
    index=False
)

Training : Logistic Regression

Accuracy : 0.9907
Precision: 0.9848
Recall   : 0.9971
F1 Score : 0.9909

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      3309
           1       0.98      1.00      0.99      3435

    accuracy                           0.99      6744
   macro avg       0.99      0.99      0.99      6744
weighted avg       0.99      0.99      0.99      6744


Confusion Matrix
[[3256   53]
 [  10 3425]]
Training : Naive Bayes

Accuracy : 0.9871
Precision: 0.9827
Recall   : 0.9921
F1 Score : 0.9874

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3309
           1       0.98      0.99      0.99      3435

    accuracy                           0.99      6744
   macro avg       0.99      0.99      0.99      6744
weighted avg       0.99      0.99      0.99      6744


Confusion Matrix
[[3249   60]
 [  27 3408]]
Training :

c:\Users\Abc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Accuracy : 0.9838
Precision: 0.9746
Recall   : 0.9942
F1 Score : 0.9843

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3309
           1       0.97      0.99      0.98      3435

    accuracy                           0.98      6744
   macro avg       0.98      0.98      0.98      6744
weighted avg       0.98      0.98      0.98      6744


Confusion Matrix
[[3220   89]
 [  20 3415]]
Training : XGBoost

Accuracy : 0.9810
Precision: 0.9680
Recall   : 0.9956
F1 Score : 0.9816

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      3309
           1       0.97      1.00      0.98      3435

    accuracy                           0.98      6744
   macro avg       0.98      0.98      0.98      6744
weighted avg       0.98      0.98      0.98      6744


Confusion Matrix
[[3196  113]
 [  15 3420]]
Training : CatBoost


KeyboardInterrupt: 